In [1]:
from pathlib import Path
import sys
import warnings
import os
import gc
from copy import deepcopy

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.ops import MLP
import torch.optim as optim
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, StratifiedKFold
from torch.utils.data import Dataset, TensorDataset, DataLoader, Subset
import optuna
import joblib
from optuna.samplers import TPESampler

import random
import math
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
import anndata
from anndata import AnnData
import pickle

sys.path.insert(0, "../../")

import scgpt as scg
from scgpt.tokenizer import GeneVocab
from scgpt import logger

warnings.filterwarnings("ignore", category=ResourceWarning)

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    
seed = 13
seed_everything(seed)

/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/allen/programs/mindscope/workgroups/auto-model/harshil.sharma/TransformerEphysPrediction/Human/Supplement/../../scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/allen/programs/mindscope/workgroups/auto-model/harshil.sharma/TransformerEphysPrediction/Human/Supplement/../../scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [2]:
# get adata for L2/3 dataset
l23_transcription_df = pd.read_csv("../Data Preprocessing/human_patchseq_L23_transcriptomicdata.csv", index_col=0)  # 50281 rows x 385 columns, rows are genes, columns are cells
l23_adata = AnnData(X=np.log1p(l23_transcription_df.values.T),obs=pd.DataFrame(index=l23_transcription_df.columns), var=pd.DataFrame(index=l23_transcription_df.index))  # AnnData wants cells as rows & genes in columns

l23_anno_df = pd.read_csv("../Data Preprocessing/human_patchseq_L23_annoData.csv")  # currently cells are named with "sample_id" column in this csv & in adata, we need "SpecimenID" for ephys data extraction
l23_adata.obs["SpecimenID"] = l23_anno_df["SpecimenID"].values

# get adata for L1 dataset
l1_transcription_df = pd.read_csv("../Data Preprocessing/human_patchseq_L1_transcriptomicData.csv", index_col=0)  # 50281 rows x 404 columns, rows are genes, columns are cells
l1_adata = AnnData(X=np.log1p(l1_transcription_df.values.T),obs=pd.DataFrame(index=l1_transcription_df.columns), var=pd.DataFrame(index=l1_transcription_df.index))  # AnnData wants cells as rows & genes in columns

l1anno_df = pd.read_csv("../Data Preprocessing/human_patchseq_L1_annoData.csv", index_col=0)  # currently cells are named with "sample_id" column in this csv & in adata, we need "spec_id_label" for ephys data extraction
l1_adata.obs["SpecimenID"] = l1anno_df["spec_id_label"].values

# combine both adata objects
combined_adata = anndata.concat([l23_adata, l1_adata], axis=0, join="outer", merge="unique")

# now select for the subset of our 50281 genes that are actually in scGPT vocabulary
model_dir = "../../scGPT_human"
model_dir = Path(model_dir)
vocab_file = model_dir / "vocab.json"

vocab = GeneVocab.from_file(vocab_file)
combined_adata.var["id_in_vocab"] = [
    vocab[gene] if gene in vocab else -1 for gene in combined_adata.var.index
]
gene_ids_in_vocab = np.array(combined_adata.var["id_in_vocab"])
combined_adata = combined_adata[:, combined_adata.var["id_in_vocab"] >= 0]
combined_adata.var.drop(columns=["id_in_vocab"], inplace=True)

# get the ephys data
# first for L2/3
l23_ephys_df = pd.read_csv("../Data Preprocessing/patchseq_L23_ephys.csv") 
human_l23_specimen_ids = l23_adata.obs["SpecimenID"]
l23_ephys_df = l23_ephys_df[l23_ephys_df["specimen_id"].isin(human_l23_specimen_ids)]  # l23_ephys_df has both human and mouse specimens, so filter by SpecimenID in adata

# now for L1
l1_ephys_df = pd.read_csv("../Data Preprocessing/human_patchseq_L1_ephys.csv")
human_l1_specimen_ids = l1_adata.obs["SpecimenID"]
l1_ephys_df = l1_ephys_df[l1_ephys_df["cell_name"].isin(human_l1_specimen_ids)]  # filter for cells with transcriptomic data
l1_ephys_df = l1_ephys_df.rename(columns={"cell_name": "specimen_id"})  # for easier concatenating of both ephys dataframes

# the l1_ephys_df has columns (parameters) not in the l23_ephys_df, so filter those out
ephys_params_in_common = [col for col in l1_ephys_df.columns if col in l23_ephys_df.columns]
l1_ephys_df = l1_ephys_df[ephys_params_in_common]
l23_ephys_df = l23_ephys_df[ephys_params_in_common]

# combine the two ephys df
combined_ephys_df = pd.concat([l1_ephys_df, l23_ephys_df], ignore_index=True)
combined_ephys_df = combined_ephys_df.dropna(how="any")  # drop NaN containing rows

# filter for cells in adata that actually have ephys data
combined_ephys_df_ids = combined_ephys_df["specimen_id"]
mask = combined_adata.obs["SpecimenID"].isin(combined_ephys_df_ids)
combined_adata = combined_adata[mask].copy()

# combined_ephys_df has "specimen_id" in a different row ordering than combined_adata's "SpecimenID," we need them to be the same:
combined_ephys_df = combined_ephys_df.set_index("specimen_id")
combined_ephys_df = combined_ephys_df.reindex(combined_adata.obs["SpecimenID"])
combined_ephys_df = combined_ephys_df.reset_index()

combined_ephys_df.drop(columns=["latency_rheo"], inplace=True)  # poor predictability

# rename ephys features from raw_name to full_name
ephys_feature_names = pd.read_csv("../Data Preprocessing/ephys_feature_names.csv")
name_mapping = dict(zip(ephys_feature_names["raw_name"], ephys_feature_names["full_name"]))
combined_ephys_df.rename(columns=name_mapping, inplace=True)

# label the top 512 highly variable genes
sc.pp.highly_variable_genes(combined_adata, n_top_genes=512)

# Get one-hot encodings for each cell based on their Seurat cell type
l1_celltypes_df = pd.read_csv("../Data Preprocessing/human_patchseq_L1_metadata.csv")
l23_celltypes_df = pd.read_csv("../Data Preprocessing/human_patchseq_L23_metadata.csv")
celltypes_df = pd.concat([l1_celltypes_df, l23_ephys_df], ignore_index=True)
l1_celltypes_df = l1_celltypes_df.rename(columns={"spec_id": "SpecimenID", "seurat_cluster": "SeuratMapping"})  # to match the column names of interest in l23_celltypes_df

# only take Seurat cell type and SpecimenID columns
l1_celltypes_df = l1_celltypes_df[["SpecimenID", "SeuratMapping"]]
l23_celltypes_df = l23_celltypes_df[["SpecimenID", "SeuratMapping"]]
celltypes_df = pd.concat([l1_celltypes_df, l23_celltypes_df], ignore_index=True)

celltypes_df = celltypes_df[celltypes_df["SpecimenID"].isin(combined_adata.obs["SpecimenID"])]  # filter for the cells we are using

# drop cells which are in cell types that have less than 4 cells, and update ephys data & adata accordingly
cell_counts_per_type = celltypes_df["SeuratMapping"].value_counts()
valid_cells = cell_counts_per_type[cell_counts_per_type >= 4].index

celltypes_df_filtered = celltypes_df[celltypes_df["SeuratMapping"].isin(valid_cells)]

dropped = celltypes_df[~celltypes_df["SeuratMapping"].isin(valid_cells)]
dropped_specimen_ids = dropped["SpecimenID"].tolist()

combined_ephys_df = combined_ephys_df[~combined_ephys_df["SpecimenID"].isin(dropped_specimen_ids)]
print(combined_ephys_df)
print("-"*125)
combined_adata = combined_adata[~combined_adata.obs["SpecimenID"].isin(dropped_specimen_ids)].copy()

specimen_to_celltype = celltypes_df_filtered.set_index("SpecimenID")["SeuratMapping"].to_dict()  # make it a dictionary of SpecimenID : cell type

unique_celltypes = celltypes_df_filtered["SeuratMapping"].unique().tolist()  # this list will be used to map a cell's type to a one hot vector (follows the order of this list)
print(unique_celltypes)
print("Number of Unique Seurat Cell Types in our set of cells (list above):", len(unique_celltypes))
print("-"*125)

# Generate one-hot cell types
encoding_matrix = np.zeros((combined_adata.n_obs, len(unique_celltypes)), dtype=int)
for i, SpecimenID in enumerate(combined_adata.obs["SpecimenID"]):
    cell_type = specimen_to_celltype[SpecimenID]
    # Find the index of the cell type in the ordered list and set the corresponding position to 1
    cell_type_index = unique_celltypes.index(cell_type)
    encoding_matrix[i, cell_type_index] = 1
combined_adata.obsm["OneHotCellType"] = encoding_matrix

types = []
for SpecimenID in combined_adata.obs["SpecimenID"]:
    types.append(specimen_to_celltype[SpecimenID])
combined_adata.obs["CellType"] = types

# percentile the combined_ephys_df values by column (expect for SpecimenID which should be unchanged)
raw_ephys_df = combined_ephys_df.copy()
cols_to_standardize = combined_ephys_df.columns.difference(["SpecimenID"])
combined_ephys_df[cols_to_standardize] = combined_ephys_df[cols_to_standardize].rank(pct=True)

print(combined_adata)

     SpecimenID       sag  resting membrane potential    rheobase  f-I slope  \
0     541549258  0.031193                  -68.778985  170.000000   0.004053   
1     541557114  0.043156                  -70.443735  220.000000   0.021537   
2     569835804  0.055736                  -80.312557  110.000000   0.073350   
3     571511167  0.090862                  -72.592708   70.000000   0.118730   
4     571640763  0.013199                  -60.348639   40.000000   0.180672   
..          ...       ...                         ...         ...        ...   
493   832712819  0.317361                  -63.053950   29.999998   0.352941   
494   868240930  0.050562                  -67.351589   20.000000   0.595000   
496   595597625  0.067271                  -68.427370   10.000000   0.422586   
497   611610593  0.052174                  -60.569354   10.000000   0.421845   
499   880463055  0.152484                  -66.865382   59.999996   0.220000   

     input resistance  time constant τ 

In [3]:
torch.cuda.set_per_process_memory_fraction(1.0, device=0)

In [4]:
# use scgpt to compute cell embeddings, updates combined_adata to cell_embeddings_adata with cell_embeddings_adata.obsm["X_scGPT"] containing the cell embeddings
cell_embeddings_adata = scg.tasks.embed_data(
    combined_adata,
    model_dir,
    gene_col="index",
    max_length=30683,
    batch_size=5,
    use_fast_transformer=False
)

scGPT - INFO - match 30682/30682 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 99/99 [02:16<00:00,  1.38s/it]
/allen/programs/mindscope/workgroups/auto-model/harshil.sharma/TransformerEphysPrediction/Human/Supplement/../../scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


In [5]:
print(cell_embeddings_adata)
print("-"*125)
print(combined_ephys_df)

AnnData object with n_obs × n_vars = 495 × 30682
    obs: 'SpecimenID', 'CellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'index', 'id_in_vocab'
    uns: 'hvg'
    obsm: 'OneHotCellType', 'X_scGPT'
-----------------------------------------------------------------------------------------------------------------------------
     SpecimenID       sag  resting membrane potential  rheobase  f-I slope  \
0     541549258  0.206061                    0.379798  0.874747   0.014141   
1     541557114  0.278788                    0.303030  0.935354   0.034343   
2     569835804  0.389899                    0.046465  0.696970   0.137374   
3     571511167  0.543434                    0.214141  0.546465   0.321212   
4     571640763  0.078788                    0.797980  0.400000   0.519192   
..          ...       ...                         ...       ...        ...   
493   832712819  0.937374                    0.676768  0.294949   0.769697   
494   868240930  0.

In [6]:
scgpt_embeddings_tensor = torch.tensor(cell_embeddings_adata.obsm["X_scGPT"], dtype=torch.float32)
ephys_tensor = torch.tensor(combined_ephys_df.drop(columns=["SpecimenID"]).values, dtype=torch.float32)
scgpt_dataset = TensorDataset(scgpt_embeddings_tensor, ephys_tensor)
datasets = {"wholehumanscgpt": scgpt_dataset}

def train_test_MLP(input_layer_size, train_dataloader, test_dataloader, num_epochs, lr, dropout, weight_decay):
    model = MLP(input_layer_size, [512, 512, 8], activation_layer=nn.Tanh, dropout=dropout)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_losses = []
    test_losses = []
    r2_scores = []

    for epoch in range(num_epochs):
        # training
        model.train()
        train_loss = 0.0
        for data_batch, targets_batch in train_dataloader:
            data_batch, targets_batch = data_batch.to(device), targets_batch.to(device)
            optimizer.zero_grad()
            outputs = model(data_batch)
            loss = criterion(outputs, targets_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()*data_batch.size(0)
        train_loss /= len(train_dataloader.sampler)
        train_losses.append(train_loss)

        # testing
        model.eval()
        test_loss = 0.0
        all_outputs = []
        all_targets = []
        with torch.no_grad():
            for data_batch, targets_batch in test_dataloader:
                data_batch, targets_batch = data_batch.to(device), targets_batch.to(device)
                outputs = model(data_batch)
                loss = criterion(outputs, targets_batch)
                test_loss += loss.item()*data_batch.size(0)
                all_outputs.append(outputs)
                all_targets.append(targets_batch)
            test_loss /= len(test_dataloader.sampler)
            test_losses.append(test_loss)

        all_outputs = torch.cat(all_outputs, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        all_outputs, all_targets = all_outputs.cpu(), all_targets.cpu()
        r2_scores.append(r2_score(all_targets, all_outputs, multioutput="raw_values")) # r2_scores is a list of num_epochs numpy arrays each 8 long (for each ephys feature)
        
    return model, all_outputs, all_targets, train_losses, test_losses, r2_scores

# for nested cross-validation loop
def make_objective(trainval_idx, dataset, input_layer_size, num_epochs, batch_size, seed):
    def objective(trial):
        # hyperparameter space
        lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        inner_scores = []

        labels = cell_embeddings_adata.obs["CellType"].iloc[trainval_idx].values

        for sub_train_idx, val_idx in inner_cv.split(trainval_idx, labels):
            sub_train_dataset = Subset(dataset, trainval_idx[sub_train_idx])
            val_dataset = Subset(dataset, trainval_idx[val_idx])

            sub_train_loader = DataLoader(sub_train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

            _, _, _, _, val_losses, r2_scores = train_test_MLP(
                input_layer_size=input_layer_size,
                train_dataloader=sub_train_loader,
                test_dataloader=val_loader,
                num_epochs=num_epochs,
                lr=lr,
                dropout=dropout,
                weight_decay=weight_decay
            )
            inner_scores.append(val_losses[-1])
        return np.mean(inner_scores)
    return objective


In [7]:
models = ["wholehumanscgpt"]
input_layer_sizes = [512]
input_layer_sizes = dict(zip(models, input_layer_sizes))

best_hyperparameters = {}
num_epochs = 1500
batch_size = 25

shuffled_cell_indices=[]

results_to_store = {"predictions": [], "train_losses": [], "test_losses": [], "r2_scores": []}
results = {model: deepcopy(results_to_store) for model in models}

ephys_truth=[]


outer_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
optuna.logging.set_verbosity(optuna.logging.FATAL)
sampler = TPESampler(seed=seed)

for outer_fold, (trainval_idx, test_idx) in enumerate(outer_cv.split(np.arange(cell_embeddings_adata.n_obs), np.array(cell_embeddings_adata.obs["CellType"]))):
    shuffled_cell_indices.append(test_idx)

    for model in models:
        # HYPERPARAMETER TUNING:
        objective = make_objective(
            trainval_idx=trainval_idx,
            dataset=datasets[model],
            input_layer_size=input_layer_sizes[model],
            num_epochs=num_epochs,
            batch_size=batch_size,
            seed=seed
        )
        study = optuna.create_study(direction="minimize", sampler=sampler)
        study.optimize(objective, n_trials=100, n_jobs=-1, gc_after_trial=True, catch=(ValueError,))
        study_file_name = f"Models/outerfold{outer_fold}_wo_types_{model}MLP_tuningstudy.pkl"
        joblib.dump(study, study_file_name)
        best_hyperparameters[model] = study.best_params

        # FINAL TRAINING ON FULL TRAINING SET, USING BEST HYPERPARAMETERS:
        train_subset = Subset(datasets[model], trainval_idx)
        test_subset = Subset(datasets[model], test_idx)

        train_dataloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
        test_dataloader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

        network, predict_subset, ephys_truth_subset, train_losses, test_losses, r2_scores = train_test_MLP(
            input_layer_size=input_layer_sizes[model],
            train_dataloader=train_dataloader,
            test_dataloader=test_dataloader,
            num_epochs=num_epochs,
            lr=best_hyperparameters[model]["lr"],
            dropout=best_hyperparameters[model]["dropout"],
            weight_decay=best_hyperparameters[model]["weight_decay"]
        )
        network.cpu()
        statedict_file_name = f"Models/outerfold{outer_fold}_wo_types_{model}MLP_weights.pth"
        torch.save(network.state_dict(), statedict_file_name)

        # store results
        results[model]["predictions"].append(predict_subset)
        results[model]["train_losses"].append(train_losses)
        results[model]["test_losses"].append(test_losses)
        results[model]["r2_scores"].append(r2_scores)

    ephys_truth.append(ephys_truth_subset)
    print(f"Outer Fold {outer_fold} Best Hyperparameters:")
    for key, value in best_hyperparameters.items():
        print(f"{key}: {value}")

Outer Fold 0 Best Hyperparameters:
wholehumanscgpt: {'lr': 0.06023026870309079, 'dropout': 0.013901623957779718, 'weight_decay': 3.442989435961985e-05}
Outer Fold 1 Best Hyperparameters:
wholehumanscgpt: {'lr': 0.08054921169383918, 'dropout': 0.031329676812608676, 'weight_decay': 0.00012406558187873053}
Outer Fold 2 Best Hyperparameters:
wholehumanscgpt: {'lr': 0.09501402101404215, 'dropout': 0.04006366260526889, 'weight_decay': 0.0003299319065773647}
Outer Fold 3 Best Hyperparameters:
wholehumanscgpt: {'lr': 0.07796789725860902, 'dropout': 0.08894243035029253, 'weight_decay': 0.00020619415509349513}


In [8]:
for model in models:
    results[model]["predictions"] = [tensor.numpy() for tensor in results[model]["predictions"]]
    results[model]["predictions"] = np.vstack(results[model]["predictions"])

    results[model]["train_losses"] = np.array(results[model]["train_losses"])
    results[model]["test_losses"] = np.array(results[model]["test_losses"])
    results[model]["r2_scores"] = np.array(results[model]["r2_scores"])

In [9]:
with open('wholehumanscgpt_results.pkl', 'wb') as f:
    pickle.dump(results, f)

In [10]:
gc.collect()
torch.cuda.empty_cache()